# *Fase de treinamento dos algoritmos*

Este notebook foi utilizado para treinar os modelos de classificação do projeto de dissertação.



In [ ]:
import pandas as pd

In [ ]:
##############################################################################
# Carregando a última versão do dataset principal
##############################################################################
dados = pd.read_csv('/content/dados2505.csv', index_col=False)
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17772 entries, 0 to 17771
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   TipoTomador                  17772 non-null  int64  
 1   ReceitaBrutaMensal           17771 non-null  float64
 2   ValorTotaBem                 17771 non-null  float64
 3   AnoEntrada                   17771 non-null  float64
 4   AnoContratacao               17771 non-null  float64
 5   MesContratacao               17771 non-null  float64
 6   QuantidadeParcelas           17771 non-null  float64
 7   TotalContratacoes            17771 non-null  float64
 8   TotalContratacoesAtraso      17771 non-null  float64
 9   ValorContratado              17771 non-null  float64
 10  AtividadeRenda               17771 non-null  float64
 11  CodigoIBGE                   17771 non-null  float64
 12  Alvo                         17771 non-null  float64
 13  IBC             

In [ ]:
##############################################################################
# Removendo inicialmente alguns campos que não ser usados nos primeiros
# treinamentos
##############################################################################
dados = dados.drop(['CodigoIBGE', 'TipoTomador','IBC','IDHM', 'participacao', 'MediaMovelEmpregabilidade'  ], axis=1)
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17772 entries, 0 to 17771
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ReceitaBrutaMensal           17771 non-null  float64
 1   ValorTotaBem                 17771 non-null  float64
 2   AnoEntrada                   17771 non-null  float64
 3   AnoContratacao               17771 non-null  float64
 4   MesContratacao               17771 non-null  float64
 5   QuantidadeParcelas           17771 non-null  float64
 6   TotalContratacoes            17771 non-null  float64
 7   TotalContratacoesAtraso      17771 non-null  float64
 8   ValorContratado              17771 non-null  float64
 9   AtividadeRenda               17771 non-null  float64
 10  Alvo                         17771 non-null  float64
 11  TempoRelacionamento          17771 non-null  float64
 12  TaxaAtrasoAjustada           17771 non-null  float64
 13  ParcelaMensalEst

In [ ]:
##############################################################################
# Esta função tem como finalidade treinar um modelo com base em um dos
# algoritmos selecionados
# Argumentos de entrada:
#         - dados é um objeto do tipo dataframe
#         - coluna_alvo é o nome da variável target do dataset
#         - modelo é um objeto do tipo DecisionTreeClassifier, RandomForestClassifier
#           ou XGBClassifier
#
##############################################################################
def executar_modelo_cv(
    dados,
    coluna_alvo='Alvo',
    modelo=None,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
):
    import pandas as pd
    import numpy as np

    from sklearn.model_selection import StratifiedKFold
    from sklearn.base import clone
    from sklearn.metrics import classification_report, confusion_matrix

    if modelo is None:
        raise ValueError("Informe um modelo válido no parâmetro 'modelo'.")

    X = dados.drop(coluna_alvo, axis=1)
    y = dados[coluna_alvo]

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    y_reais = []
    y_preditos = []
    y_probabilidades = []

    print(f"\nValidação cruzada com {n_splits} folds")
    print(f"SMOTE aplicado: {usar_smote}")

    for fold, (idx_treino, idx_teste) in enumerate(skf.split(X, y), start=1):

        print(f"\n===== Fold {fold} =====")

        X_treino = X.iloc[idx_treino]
        X_teste = X.iloc[idx_teste]
        y_treino = y.iloc[idx_treino]
        y_teste = y.iloc[idx_teste]

        print("\nDistribuição no treino antes do balanceamento:")
        print(y_treino.value_counts())

        if usar_smote:
            from imblearn.over_sampling import SMOTE

            smote = SMOTE(random_state=random_state)
            X_treino, y_treino = smote.fit_resample(X_treino, y_treino)

            print("\nDistribuição no treino após SMOTE:")
            print(pd.Series(y_treino).value_counts())

        modelo_fold = clone(modelo)
        modelo_fold.fit(X_treino, y_treino)

        y_pred = modelo_fold.predict(X_teste)

        y_reais.extend(y_teste)
        y_preditos.extend(y_pred)

        if hasattr(modelo_fold, "predict_proba"):
            y_proba = modelo_fold.predict_proba(X_teste)[:, 1]
            y_probabilidades.extend(y_proba)

    if len(y_probabilidades) == len(y_reais):
        y_proba_final = np.array(y_probabilidades)
    else:
        y_proba_final = None

    print("\n===== RESULTADO CONSOLIDADO DA VALIDAÇÃO CRUZADA =====\n")
    print(classification_report(y_reais, y_preditos, zero_division=0))

    print("\nMatriz de confusão consolidada:")
    print(confusion_matrix(y_reais, y_preditos, labels=[0, 1]))

    metricas = calcular_metricas(
        y_real=np.array(y_reais),
        y_pred=np.array(y_preditos),
        y_proba=y_proba_final,
        pos_label=pos_label
    )

    print("\nMétricas consolidadas:")
    print(metricas)

    modelo_final = clone(modelo)

    X_final = X.copy()
    y_final = y.copy()

    if usar_smote:
        from imblearn.over_sampling import SMOTE

        smote = SMOTE(random_state=random_state)
        X_final, y_final = smote.fit_resample(X_final, y_final)

    modelo_final.fit(X_final, y_final)

    return (
        modelo_final,
        np.array(y_reais),
        np.array(y_preditos),
        metricas
    )

In [ ]:
##############################################################################
# Função usada para automatizar a geração das métricas. A versão anterior
# apresentava um erro ao executa a accuracy_score e precisou ser
# modificada e nesta versão está diferente da documentação do desenvolvedor
# Argumentos de entrada:
#
##############################################################################
def calcular_metricas(y_real, y_pred, y_proba=None, pos_label=1):
    import pandas as pd
    from sklearn.metrics import (
        accuracy_score,
        recall_score,
        precision_score,
        f1_score,
        roc_auc_score,
        confusion_matrix
    )

    tn, fp, fn, tp = confusion_matrix(
        y_real, y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp)

    metricas = {
        "Accuracy": accuracy_score(y_real, y_pred),
        "Recall": recall_score(y_real, y_pred, pos_label=pos_label, zero_division=0),
        "Precision": precision_score(y_real, y_pred, pos_label=pos_label, zero_division=0),
        "Specificity": specificity,
        "F1-Score": f1_score(y_real, y_pred, pos_label=pos_label, zero_division=0),
        "ROC-AUC": roc_auc_score(y_real, y_proba) if y_proba is not None else None
    }

    return pd.DataFrame([metricas])

Treinamento do modelo com Decision Tree, amostra completa e sem ajuste de hiperparâmetro

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(random_state=52)

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Agora Treinando o algoritmo Random Forest sem ajustes de hiperparâmetros e com a amostra completa

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Agora o mesmo para XGBoost sem ajustes nos hiperparâmetros e com a amostra completa

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Agora repetindo o mesmo processo porém utilizando os Hiperparâmetros selecionados para cada algoritmo

Treinamento do modelo com Decison Tree tendo os hiperparâmetros ajustados e com dados completos


In [ ]:
#################################################################
# Algoritmo Decision Tree após ajustes de hiperparâmetros
#################################################################
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Treinamento do modelo com Random Forest tendo os hiperparâmetros ajustados e com dados completos

In [ ]:
#################################################################
# Random forest após ajustes nos hiperparâmetros
#################################################################
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Treinamento do modelo com XGBoost tendo os hiperparâmetros com a base completa

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Gerando dois datasets em separado uma para cada tipo de tomador com base em
1 - Pessoa Física
2 - Pessoa jurídica

In [ ]:
#########################################################################
# Este trecho estava em outro notebook e foi migrado para este mas
# a mesma estrutura que coincidência ficou semelhante ao código no começo
#########################################################################
dados  = pd.read_csv('/content/dados2505.csv', index_col=False)
dados = dados.drop(['CodigoIBGE', 'IBC','IDHM', 'participacao', 'MediaMovelEmpregabilidade'  ], axis=1)


# Pessoa Física
dadosPF = dados[dados['TipoTomador'] == 1].copy()


# Pessoa Jurídica
dadosPJ = dados[dados['TipoTomador'] == 2].copy()

Modelo treinado com Decison Tree sem ajustes de hiperparâmetros e com dados de pessoa física

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(random_state=52)

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPF,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Modelo treinado com Decision Tree sem ajustes de hiperparâmetros e com dados de pessoa jurídica



In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(random_state=52)

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPJ,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Modelo treinado com Random Forest sem ajustes de hiperparâmetros e com dados de pessoa Física

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPF,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Modelo treinado com Random Forest sem ajustes de hiperparâmetros e com dados de pessoa jurídica

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPJ,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Modelo treinado com XGBoost sem ajustes de hiperparâmetros e com dados de pessoa física

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPF,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Modelo treinado com XGBoost sem ajustes de hiperparâmetros e com dados de pessoa jurídica

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPJ,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Treinamento do modelo com Decison Tree tendo os hiperparâmetros ajustados e com a amostra completa


In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Treinamento do modelo com Decison Tree tendo os hiperparâmetros ajustados e com dados de Pessoa Física

In [ ]:
#################################################################
# Algoritmo Decision Tree após ajustes de hiperparâmetros
# apenas para Pessoas Físicas
#################################################################
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPF,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Treinamento do modelo com Decision Tree tendo os hiperparâmetros ajustados e com dados de Pessoa Jurídica

In [ ]:
#################################################################
# Algoritmo Decision Tree após ajustes de hiperparâmetros
# apenas para Pessoas Jurídicas
#################################################################
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPJ,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Treinamento do modelo com Random Forest tendo os hiperparâmetros e com a amostra completa

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Treinamento do modelo com Random Forest tendo os hiperparâmetros ajustados e com dados de Pessoa Física

In [ ]:
#################################################################
# Random forest após ajustes nos hiperparâmetros
# Apenas para Pessoas Físicas
#################################################################
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPF,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Treinamento do modelo com Random Forest tendo os hiperparâmetros ajustados e com dados de Pessoa Jurídica

In [ ]:
#################################################################
# Random forest após ajustes nos hiperparâmetros
# Apenas para Pessoas Físicas
#################################################################
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPJ,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Treinamento do modelo com XGBoost tendo os hiperparâmetros ajustados e com toda amostra

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dados,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Treinamento do modelo com XGBoost tendo os hiperparâmetros ajustados e com dados de Pessoa Física

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPF,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Treinamento do modelo com XGBoost tendo os hiperparâmetros ajustados e com dados de Pessoa Jurídica

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosPJ,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

A partir de agora adicionando as variáveis socioeconômicas e repetindo os experimentos

In [ ]:
dadosFinais = pd.read_csv('/content/dados2505.csv', index_col=False)
dadosFinais = dadosFinais.drop(['CodigoIBGE', ], axis=1)
dadosFinais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21352 entries, 0 to 21351
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   TipoTomador                  21352 non-null  int64  
 1   ReceitaBrutaMensal           21352 non-null  float64
 2   ValorTotaBem                 21352 non-null  float64
 3   AnoEntrada                   21352 non-null  int64  
 4   AnoContratacao               21352 non-null  int64  
 5   MesContratacao               21352 non-null  int64  
 6   QuantidadeParcelas           21352 non-null  int64  
 7   TotalContratacoes            21352 non-null  int64  
 8   TotalContratacoesAtraso      21352 non-null  int64  
 9   ValorContratado              21352 non-null  float64
 10  AtividadeRenda               21352 non-null  int64  
 11  Alvo                         21352 non-null  int64  
 12  IBC                          21352 non-null  float64
 13  IDHM            

Primeiro treinando todos os algoritmos com as amostra completas sem separar pessoas físicas de jurídicas

Modelo com Decision Tree com amostra completa e com dados socioeconômicos

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinais,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Modelo com Random Forest com amostra completa e com dados socioeconômicos

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinais,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

Modelo com XGBOost com amostra completa e com dados socioeconômicos

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinais,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11412
1     5670
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    11412
1    11412
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    11411
0    11411
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    11411
1     5671
Name: count, 

In [ ]:

# Pessoa Física
dadosFinaisPF = dadosFinais[dadosFinais['TipoTomador'] == 1].copy()


# Pessoa Jurídica
dadosFinaisPJ = dadosFinais[dadosFinais['TipoTomador'] == 2].copy()

Treinando modelo final com Decision Tree e amostra de pessoa física

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinaisPF,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)



Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Treinando modelo final com Decision Tree e amostra de pessoa jurídica

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modeloDT = DecisionTreeClassifier(
                                  random_state=52
                                  ,ccp_alpha=0.016685430556951094
                                  ,class_weight='balanced'
                                  ,criterion='log_loss'
                                  ,max_depth=24
                                  ,max_features=None
                                  ,max_leaf_nodes=10
                                  ,min_impurity_decrease=0.03609993861334124
                                  ,min_samples_leaf=6
                                  ,min_samples_split=3
                                  ,min_weight_fraction_leaf=0.018182496720710064
                                  ,splitter='best'
                                  )

modeloDT, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinaisPJ,
    coluna_alvo='Alvo',
    modelo=modeloDT,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)



Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Treinando modelo final com Random Forest e amostra de pessoa física

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinaisPF,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Treinando modelo final com Random Forest e amostra de pessoa jurídica

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modeloRF = RandomForestClassifier(
    random_state=52
    ,bootstrap=True
    ,ccp_alpha=0.022524962598477152
    ,class_weight='balanced'
    ,criterion='gini'
    ,max_depth=29
    ,max_features='log2'
    ,max_leaf_nodes=170
    ,max_samples=0.692708251269958
    ,min_impurity_decrease=0.0007983126110107097
    ,min_samples_leaf=2
    ,min_samples_split=21
    ,min_weight_fraction_leaf=0.024102546602601173
    ,n_estimators=554
)

modeloRF, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinaisPJ,
    coluna_alvo='Alvo',
    modelo=modeloRF,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist

Treinando modelo final com XGBoost e amostra de pessoa física

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinaisPF,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4974
0    4974
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4974
1    2510
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    4974
1    4974
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
1    4973
0    4973
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    4973
1    2511
Name: count, dtype: int64

Dist

Treinando modelo final com XGBoost e amostra de pessoa Jurídica

In [ ]:
from xgboost import XGBClassifier

modeloXGB = XGBClassifier(
    random_state=52
    ,colsample_bylevel= 0.7995146823886632
    ,colsample_bynode= 0.9133994139045392
    ,colsample_bytree= 0.9795373972820138
    ,gamma= 3.425242571785434
    ,grow_policy= 'lossguide'
    ,learning_rate= 0.009839149612911786
    ,max_bin= 366
    ,max_depth= 11
    ,max_leaves= 45
    ,min_child_weight= 11
    ,n_estimators= 495
    ,reg_alpha= 2.958792117688386
    ,reg_lambda= 6.834349194024739e-05
    ,scale_pos_weight= 3.01589206031773
    ,subsample= 0.628007765926831
    ,tree_method= 'hist'
)

modeloXGB, y_teste, y_pred, metricas = executar_modelo_cv(
    dados=dadosFinaisPJ,
    coluna_alvo='Alvo',
    modelo=modeloXGB,
    usar_smote=True,
    n_splits=5,
    random_state=42,
    pos_label=1
)


Validação cruzada com 5 folds
SMOTE aplicado: True

===== Fold 1 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 2 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6437
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6437
1    6437
Name: count, dtype: int64

===== Fold 3 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 4 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Distribuição no treino após SMOTE:
Alvo
0    6438
1    6438
Name: count, dtype: int64

===== Fold 5 =====

Distribuição no treino antes do balanceamento:
Alvo
0    6438
1    3160
Name: count, dtype: int64

Dist